### Initialize the environement

In [1]:
from envs import my_matc_env_rb_strong

RENDER_MODE = 'sim'

# Initializing the environment
flight_route_file = 'routes/matc/auto_airspace_config.json'
aircraft_config_file = 'routes/matc/aircraft_configurations_auto.json'
env = my_matc_env_rb_strong.MATCEnv(flight_route_file, aircraft_config_file, render_mode=RENDER_MODE)

Using Python-based geo functions
Successfully loaded OpenAP performance model
Failed to load BADA performance model
Successfully loaded legacy performance model
Reading config from /Users/imansharifi/Documents/VirtualEnvs/ReinforcementLearning/Code/ATC_LLM/settings.cfg
Reading magnetic variation data
Loading global navigation database...
Reading cache: /Users/imansharifi/Documents/VirtualEnvs/ReinforcementLearning/Code/ATC_LLM/cache/navdata.p
Successfully loaded plugin AREA
Successfully loaded plugin DATAFEED


## Convert the raw ownship-intruder information to the prompts with answers

In [15]:
def get_ownship_prompt_v1(env, ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action']

    prompt = "You are an airspace tactical deconfliction assistant. " + \
            "Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by suggesting appropriate speed adjustments. " + \
            "This task is a safety-critical task, requiring careful consideration of the situation. " + \
            "Front intruders are aircraft that are ahead of the ownship and closer to the next waypoint, no matter on the same route or a different route. " + \
            f"At each time step, the ownship can choose one of the following actions: 'Accelerate' (increase speed by {ownship_info['speed_change_per_second(m/s2)']} m/s per second), " + \
            f"'Decelerate' (decrease speed by -{ownship_info['speed_change_per_second(m/s2)']} m/s per second), or 'Hold' its current speed. " + \
            f"The intruders are ahead of the ownship, but there may be other intruders behind the ownship. " + \
            f"The ownship cannot unnecessarily decelerate since it might occlude the airspace for the intruders behind it." + \
            f"The simulation time step is {env.dt} seconds. \n" + \
            "Based on the following information about the ownship and its intruders, recommend one action to adjust " + \
            "its current speed to maintain safe separation. \nHere is the ownship information: \n"

    # prompt += f"Ownship ID: {ownship_info['agent_id']}, "
    prompt += f"Aircraft Type: {ownship_info['ac_type']}, "
    # prompt += f"Latitude: {ownship_info['ac_lat']}, Longitude: {ownship_info['ac_lon']}, "
    prompt += f"Next Waypoint ID: {ownship_info['next_wpt_id']}, Next Waypoint Type: {ownship_info['next_wpt_type']}, "
    prompt += f"Distance to Next Waypoint (m): {ownship_info['ac_dist_to_nxt_wpt(m)']}, "
    prompt += f"Speed (m/s): {ownship_info['speed(m/s)']}, "
    prompt += f"Speed Range (m/s): [{ownship_info['ac_min_spd(m/s)']}, {ownship_info['ac_max_spd(m/s)']}], "
    prompt += f"Speed Change per Second (m/s²): {ownship_info['speed_change_per_second(m/s2)']}, "
    # prompt += f"Acceleration (knot/s): {ownship_info['acceleration(knot/s)']}, "
    prompt += f"Heading (deg): {ownship_info['heading(deg)']} with heading scale {ownship_info['heading_scale']}, "
    # prompt += f"Altitude (ft): {ownship_info['altitude(ft)']}, "
    # prompt += f"Route ID: {ownship_info['ac_route_id']}, "
    prompt += f"Last Action: {ownship_info['ac_last_action']}, "
    
    safety_distance = env.get_agent_safety_distance(ownship_info['agent_id']) * 1852  # convert from nautical miles to meters
    ownship_collision_dist = env.get_agent_collision_distance(ownship_info['agent_id']) * 1852  # convert from nautical miles to meters
    # prompt += f"The safety distance to maintain from intruders is {safety_distance:.2f} (m), which is a safe boundary around the ownship. "
    prompt += f"The near mid-air collision distance is {ownship_collision_dist:.2f} (m), which is the most critical distance between the ownship and the intruder. \n"
    prompt += "Here are the intruder details: \n"
    prompt += f"Number of Intruders Ahead: {ownship_info['num_intruders_ahead']}. "
    if ownship_info['num_intruders_ahead'] == 1:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"
        
    elif ownship_info['num_intruders_ahead'] == 2:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"
        # prompt += f"Are the first intruder and the ownship on the same route? {ownship_info['intruder1_on_same_route']}. \n"
        
        prompt += "Tthe second closest front intruder information: \n"
        # prompt += f"Intruder ID: {second_intruder_info['agent_id']}, "
        prompt += f"Type: {second_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {second_intruder_info['ac_lat']}, Longitude: {second_intruder_info['ac_lon']}, "
        prompt += f"Next Waypoint ID: {second_intruder_info['next_wpt_id']}, Next Waypoint Type: {second_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {second_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {second_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {second_intruder_info['acceleration(knot/s)']}, "
        prompt += f"Heading (deg): {second_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {second_intruder_info['altitude(ft)']}, "
        # prompt += f"Route ID: {second_intruder_info['ac_route_id']}, "
        prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder2(m)']}. "
        if ownship_info['intruder2_on_same_route']:
            prompt += "The second intruder and the ownship are on the same route. \n"
        else:
            prompt += "The second intruder and the ownship are not on the same route. \n"

    else:
        prompt += "There are no front intruders detected, so try to avoid occlusion. \n"

    prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"
    prompt_response = f"The recommended action for the ownship is to '{ownship_action}'. \n"
    return prompt, prompt_response

In [26]:
def get_ownship_prompt(env, ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action']

    prompt = "You are an airspace tactical deconfliction assistant. " + \
            "Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by suggesting appropriate speed adjustments. " + \
            "This task is a safety-critical task, requiring careful consideration of the situation. " + \
            "Front intruders are aircraft that are ahead of the ownship and closer to the next waypoint, no matter on the same route or a different route. " + \
            f"At each time step, the ownship can choose one of the following actions: 'Accelerate' (increase speed by {ownship_info['speed_change_per_second(m/s2)']} m/s per second), " + \
            f"'Decelerate' (decrease speed by -{ownship_info['speed_change_per_second(m/s2)']} m/s per second), or 'Hold' its current speed. " + \
            f"The intruders are ahead of the ownship, but there may be other intruders behind the ownship. " + \
            f"The ownship cannot unnecessarily decelerate since it might occlude the airspace for the intruders behind it." + \
            f"The simulation time step is {env.dt} seconds. \n" + \
            "Based on the following information about the ownship and its intruders, recommend one action to adjust " + \
            "its current speed to maintain safe separation. \nHere is the ownship information: \n"

    # prompt += f"Ownship ID: {ownship_info['agent_id']}, "
    # prompt += f"Aircraft Type: {ownship_info['ac_type']}, "
    # prompt += f"Latitude: {ownship_info['ac_lat']}, Longitude: {ownship_info['ac_lon']}, "
    prompt += f"Speed (m/s): {ownship_info['speed(m/s)']}, "
    prompt += f"Speed Range (m/s): [{ownship_info['ac_min_spd(m/s)']}, {ownship_info['ac_max_spd(m/s)']}], "
    prompt += f"Speed Change per Second (m/s²): {ownship_info['speed_change_per_second(m/s2)']}, "
    # prompt += f"Next Waypoint ID: {ownship_info['next_wpt_id']}, Next Waypoint Type: {ownship_info['next_wpt_type']}, "
    # prompt += f"Distance to Next Waypoint (m): {ownship_info['ac_dist_to_nxt_wpt(m)']}, "

    # prompt += f"Acceleration (knot/s): {ownship_info['acceleration(knot/s)']}, "
    # prompt += f"Heading (deg): {ownship_info['heading(deg)']} with heading scale {ownship_info['heading_scale']}, "
    # prompt += f"Altitude (ft): {ownship_info['altitude(ft)']}, "
    # prompt += f"Route ID: {ownship_info['ac_route_id']}, "
    # prompt += f"Last Action: {ownship_info['ac_last_action']}, "

    # safety_distance = int(env.get_agent_safety_distance(ownship_info['agent_id']) * 1852)  # convert from nautical miles to meters
    # ownship_collision_dist = int(env.get_agent_collision_distance(ownship_info['agent_id']) * 1852)  # convert from nautical miles to meters
    # prompt += f"The safety distance to maintain from intruders is {safety_distance:.2f} (m), which is a safe boundary around the ownship. "
    # prompt += f"The near mid-air collision distance is {ownship_collision_dist:.2f} (m), which is the most critical distance between the ownship and the intruder. \n"
    prompt += "Here are the intruder details: \n"
    prompt += f"There are {ownship_info['num_intruders_ahead']} intruders ahead. "
    if ownship_info['num_intruders_ahead'] == 1:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        # prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        # prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        # prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        # prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"

        # format time to collision to 2 decimal places
        prompt += f"Ownship's time to collision with the first intruder is {ownship_info['time_to_collision_with_inruder1(s)']:.2f} (s). "

    elif ownship_info['num_intruders_ahead'] == 2:
        prompt += "The first closest front intruder information: \n"
        # prompt += f"Intruder ID: {first_intruder_info['agent_id']}, "
        # prompt += f"Type: {first_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {first_intruder_info['ac_lat']}, Longitude: {first_intruder_info['ac_lon']}, "
        # prompt += f"Next Waypoint ID: {first_intruder_info['next_wpt_id']}, Next Waypoint Type: {first_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {first_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {first_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {first_intruder_info['acceleration(knot/s)']}, "
        # prompt += f"Heading (deg): {first_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {first_intruder_info['altitude(m)']}, "
        # prompt += f"Route ID: {first_intruder_info['ac_route_id']}, "
        # prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder1(m)']}. "
        if ownship_info['intruder1_on_same_route']:
            prompt += "The first intruder and the ownship are on the same route. \n"
        else:
            prompt += "The first intruder and the ownship are not on the same route. \n"

        prompt += f"Ownship's time to collision with the first intruder is {ownship_info['time_to_collision_with_inruder1(s)']:.2f} (s). "

        prompt += "The second closest front intruder information: \n"
        # prompt += f"Intruder ID: {second_intruder_info['agent_id']}, "
        # prompt += f"Type: {second_intruder_info['ac_type']}, "
        # prompt += f"Latitude: {second_intruder_info['ac_lat']}, Longitude: {second_intruder_info['ac_lon']}, "
        # prompt += f"Next Waypoint ID: {second_intruder_info['next_wpt_id']}, Next Waypoint Type: {second_intruder_info['next_wpt_type']}, "
        prompt += f"Distance to Next Waypoint (m): {second_intruder_info['ac_dist_to_nxt_wpt(m)']}, "
        prompt += f"Speed (m/s): {second_intruder_info['speed(m/s)']}, "
        # prompt += f"Acceleration (knot/s): {second_intruder_info['acceleration(knot/s)']}, "
        # prompt += f"Heading (deg): {second_intruder_info['heading(deg)']}, "
        # prompt += f"Altitude (ft): {second_intruder_info['altitude(ft)']}, "
        # prompt += f"Route ID: {second_intruder_info['ac_route_id']}, "
        # prompt += f"Distance to the ownship (m): {ownship_info['distance_to_intruder2(m)']}. "
        if ownship_info['intruder2_on_same_route']:
            prompt += "The second intruder and the ownship are on the same route. \n"
        else:
            prompt += "The second intruder and the ownship are not on the same route. \n"

        prompt += f"Ownship's time to collision with the second intruder is {ownship_info['time_to_collision_with_inruder2(s)']:.2f} (s). "

    else:
        prompt += "There are no front intruders detected, so try to avoid occlusion. \n"

    prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"

    prompt_response = f"The recommended action for the ownship is to '{ownship_action}'. \n"
    
    return prompt, prompt_response

In [2]:
def get_ownship_prompt(env, ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action']

    system_prompt = """You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, 
    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, 
    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
    v: ownship speed(m/s), 
    v_min: ownship minimum speed(m/s), 
    v_max: ownship maximum speed(m/s),
    d1: the absolute distance between the first front intruder (if any) and ownship(m), 
    d2: the absolute distance between the second front intruder (if any) and ownship(m), 
    delta_dwpt1: the absolute difference between the distance to the next waypoint between the first front intruder (if any) and ownship(m), 
    delta_dwpt2: the absolute difference between the distance to the next waypoint between the second front intruder (if any) and ownship(m), 
    delta_v1: signed speed difference between the first front intruder and ownship(m/s), 
    delta_v2: signed speed difference between the second front intruder and ownship(m/s), 
    same_route1: whether the first front intruder is on the same route as the ownship (1: yes, 0: no), 
    same_route2: whether the second front intruder is on the same route as the ownship (1: yes, 0: no). 
    If one of the values is None, it means there is no such front intruder.
    Based on the above information, decide the ownship action: Decelerate / Hold / Accelerate. 
    Never include reasoning, explanations, or any text other than the single chosen action.
    """

    user_prompt = """Here is the ownship state information:
    [v, v_min, v_max, d1, d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2] = [{v:.2f}, {v_min:.2f}, {v_max:.2f}, {d1:.2f}, {d2:.2f}, {delta_dwpt1:.2f}, {delta_dwpt2:.2f}, {delta_v1:.2f}, {delta_v2:.2f}, {same_route1:.0f}, {same_route2:.0f}]. 
    Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate)"""
    
    ownship_spd = ownship_info['speed(m/s)']
    ownship_min_spd = ownship_info['ac_min_spd(m/s)']
    ownship_max_spd = ownship_info['ac_max_spd(m/s)']
    d1 = ownship_info['distance_to_intruder1(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    d2 = ownship_info['distance_to_intruder2(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_dwpt1 = abs(first_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_dwpt2 = abs(second_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_v1 = first_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_v2 = second_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 2 else None
    same_route1 = ownship_info['intruder1_on_same_route'] if ownship_info['num_intruders_ahead'] >= 1 else None
    same_route2 = ownship_info['intruder2_on_same_route'] if ownship_info['num_intruders_ahead'] >= 2 else None

    user_prompt = user_prompt.format(
        v=ownship_spd,
        v_min=ownship_min_spd,
        v_max=ownship_max_spd,
        d1=d1 if d1 is not None else float('nan'),
        d2=d2 if d2 is not None else float('nan'),
        delta_dwpt1=delta_dwpt1 if delta_dwpt1 is not None else float('nan'),
        delta_dwpt2=delta_dwpt2 if delta_dwpt2 is not None else float('nan'),
        delta_v1=delta_v1 if delta_v1 is not None else float('nan'),
        delta_v2=delta_v2 if delta_v2 is not None else float('nan'),
        same_route1=same_route1 if same_route1 is not None else float('nan'),
        same_route2=same_route2 if same_route2 is not None else float('nan')
    )

    prompt_response = f"The recommended action for the ownship is to '{ownship_action}'. \n"
    
    return system_prompt, user_prompt, prompt_response

In [9]:
# vehicle_speed_categories = {
#     "Amazon Prime Air - MK30 Model": ,
#     "Google X-Wing": ,
# }

def speed_indicator(speed, max_speed):
    """
    Converts a speed value into a categorical indicator: low, medium, or high.
    """
    if speed < 0.15 * max_speed:
        return 'very low'
    elif 0.15 * max_speed <= speed < 0.33 * max_speed:
        return 'low'
    elif 0.33 * max_speed <= speed < 0.55 * max_speed:
        return 'medium'
    elif 0.55 * max_speed <= speed < 0.75 * max_speed:
        return 'high'
    else:
        return 'very high'
    
def desired_speed_indicator(current_speed, max_speed):
    desired_speed = 0.75 * max_speed  # Example desired speed at 75% of max speed

    if current_speed < desired_speed:
        return 'lower'
    else:  
        return 'higher'

def distance_indicator(distance):
    """
    Converts a distance value into a categorical indicator: near, medium, or far.
    """
    safe_distance = 500  # Example maximum distance for categorization
    if distance < 0.5 * safe_distance:
        return 'too unsafe'
    elif 0.5 * safe_distance <= distance <  safe_distance:
        return 'unsafe'
    elif safe_distance <= distance < 1.5 * safe_distance:
        return 'medium safe'
    elif 1.5 * safe_distance <= distance < 2.5 * safe_distance:
        return 'safe'
    else:
        return 'very safe'
    
def wypoint_distance_indicator(delta_distance):
    """
    Converts a waypoint distance difference into a categorical indicator: close, moderate, or far.
    """
    safe_distance = 500  # Example maximum distance for categorization
    if delta_distance < 0.3 * safe_distance:
        return 'too short' # too close
    elif 0.3 * safe_distance <= delta_distance < 0.5 * safe_distance:
        return 'short' # close
    elif 0.5 * safe_distance <= delta_distance < 1.5 * safe_distance:
        return 'moderate' # moderate
    elif 1.5 * safe_distance <= delta_distance < 2.5 * safe_distance:
        return 'long' # far
    else:
        return 'very long' # so far
    
def delta_waypoint_distance_indicator(delta_distance):
    if delta_distance <= 50:
        return 'nearly equal'
    elif 50 < delta_distance <= 200:
        return 'slightly different'
    elif 200 < delta_distance <= 500:
        return 'moderately different'
    else:
        return 'significantly different'
    
def delta_speed_indicator(delta_speed, max_speed):
    if abs(delta_speed) < 0.1 * max_speed:
        return 'nearly equal'
    elif 0.1 * max_speed <= abs(delta_speed) < 0.3 * max_speed and delta_speed > 0:
        return 'slightly higher'
    elif 0.1 * max_speed <= abs(delta_speed) < 0.3 * max_speed and delta_speed < 0:
        return 'slightly lower'
    elif 0.3 * max_speed <= abs(delta_speed) < 0.6 * max_speed and delta_speed > 0:
        return 'moderately higher'
    elif 0.3 * max_speed <= abs(delta_speed) < 0.6 * max_speed and delta_speed < 0:
        return 'moderately lower'
    elif abs(delta_speed) >= 0.6 * max_speed and delta_speed > 0:
        return 'significantly higher'
    else:
        return 'significantly lower'


def get_ownship_prompt(ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action'] 

    # system_prompt = """You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, such as merging or intersections points, where other agents (intruders) are approaching as well. 
    #     Based on the information of the ownship and intruders, the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    #     but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    #     suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    #     """
    system_prompt = (
        f"You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, such as merging or intersections points, where other agents (intruders) are approaching as well.\n"
        f"Based on the information of the ownship and intruders, the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, but there might be other intruders behind the ownship. \n" 
        f"Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by suggesting appropriate speed adjustments. \n"
        f"The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it. \n"
        f"Your response should start with 'The recommended action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate.\n"
    )
        
        # Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
        # v: ownship speed(m/s). The value can be either (low, medium, or high) 
        # is_v_zero: whether the ownship speed is zero (yes, no).
        # v_min: ownship minimum speed(m/s), 
        # v_max: ownship maximum speed(m/s),
        # d1: the absolute distance between the first front intruder (if any) and ownship(m), 
        # d2: the absolute distance between the second front intruder (if any) and ownship(m), 
        # delta_dwpt1: the absolute difference between the distance to the next waypoint between the first front intruder (if any) and ownship(m), 
        # delta_dwpt2: the absolute difference between the distance to the next waypoint between the second front intruder (if any) and ownship(m), 
        # delta_v1: signed speed difference between the first front intruder and ownship(m/s), 
        # delta_v2: signed speed difference between the second front intruder and ownship(m/s), 
        # same_route1: whether the first front intruder is on the same route as the ownship (1: yes, 0: no), 
        # same_route2: whether the second front intruder is on the same route as the ownship (1: yes, 0: no). 
        # If one of the values is None, it means there is no such front intruder.
        # Based on the above information, decide the ownship action: Decelerate / Hold / Accelerate. 
        # Never include reasoning, explanations, or any text other than the single chosen action.
        # """
    
    ownship_spd = ownship_info['speed(m/s)']
    ownship_min_spd = ownship_info['ac_min_spd(m/s)']
    ownship_max_spd = ownship_info['ac_max_spd(m/s)']
    d1 = ownship_info['distance_to_intruder1(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    d2 = ownship_info['distance_to_intruder2(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    o_wpt = ownship_info['ac_dist_to_nxt_wpt(m)']
    dwpt1 = first_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    dwpt2 = second_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_dwpt1 = abs(first_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_dwpt2 = abs(second_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_v1 = first_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_v2 = second_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 2 else None
    same_route1 = ownship_info['intruder1_on_same_route'] if ownship_info['num_intruders_ahead'] >= 1 else None
    same_route2 = ownship_info['intruder2_on_same_route'] if ownship_info['num_intruders_ahead'] >= 2 else None

    def intruder_presence(num_intruders):
        if num_intruders == 0:
            return 'no'
        elif num_intruders == 1:
            return 'one'
        else:
            return 'two'
        
    user_prompt = (
        f"Given the information of the ownship and intruders as follows: \n"
        f"- Ownship: \n"
        f"  - Speed is {speed_indicator(ownship_spd, ownship_max_spd)} ({ownship_spd} m/s), where minimum possible speed is {ownship_min_spd} m/s  and maximum possible speed is {ownship_max_spd} m/s. \n"
        f"  - Speed is {desired_speed_indicator(ownship_spd, ownship_max_spd)} than the desired speed. \n"
        f"  - Speed is{' ' if ownship_spd == 0 else ' not '}minimum and is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum.\n"
        # f"- The ownship speed is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum. \n"
        f"  - Distance to the next waypoint is {wypoint_distance_indicator(o_wpt)} ({o_wpt} m).   \n"
        f"  - There are {intruder_presence(ownship_info['num_intruders_ahead'])} intruders ahead. \n"
    )

    if ownship_info['num_intruders_ahead'] >= 1:
        user_prompt += (
            f"- Front Intruder 1: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d1)} ({d1} m). \n"
            f"  - The intruder is{' ' if same_route1 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt1)} ({dwpt1} m). \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt1)} than the ownship. \n"
            f"  - The intruder speed is {first_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v1, ownship_max_spd)} speed compared to the ownship. \n"
        )    

    if ownship_info['num_intruders_ahead'] >= 2:
        user_prompt += (
            f"- Front Intruder 2: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d2)} ({d2} m). \n"
            f"  - The intruder is{' ' if same_route2 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt2)} ({dwpt2} m). \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt2)} than the ownship. \n"
            f"  - The intruder speed is {second_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v2, ownship_max_spd)} speed compared to the ownship. \n"
        )

    if ownship_info['num_intruders_ahead'] == 0:
        user_prompt += "Since there are no front intruders detected, make your decision based on the current speed and desired speed.\n"
    
    user_prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"
    user_prompt += f"Your response should start with 'The recommended action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate.\n"

    prompt_response = f"The recommended action is: {ownship_action} \n"

    return system_prompt, user_prompt, prompt_response



In [ ]:
def get_ownship_prompt_no_number(ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action'] 

    # system_prompt = """You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, such as merging or intersections points, where other agents (intruders) are approaching as well. 
    #     Based on the information of the ownship and intruders, the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    #     but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    #     suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    #     """
    system_prompt = (
        f"You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, such as merging or intersections points, where other agents (intruders) are approaching as well.\n"
        f"Based on the information of the ownship and intruders, the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, but there might be other intruders behind the ownship. \n" 
        f"Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by suggesting appropriate speed adjustments. \n"
        f"The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it. \n"
        f"Your response should start with 'The recommended action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate.\n"
    )
        
        # Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
        # v: ownship speed(m/s). The value can be either (low, medium, or high) 
        # is_v_zero: whether the ownship speed is zero (yes, no).
        # v_min: ownship minimum speed(m/s), 
        # v_max: ownship maximum speed(m/s),
        # d1: the absolute distance between the first front intruder (if any) and ownship(m), 
        # d2: the absolute distance between the second front intruder (if any) and ownship(m), 
        # delta_dwpt1: the absolute difference between the distance to the next waypoint between the first front intruder (if any) and ownship(m), 
        # delta_dwpt2: the absolute difference between the distance to the next waypoint between the second front intruder (if any) and ownship(m), 
        # delta_v1: signed speed difference between the first front intruder and ownship(m/s), 
        # delta_v2: signed speed difference between the second front intruder and ownship(m/s), 
        # same_route1: whether the first front intruder is on the same route as the ownship (1: yes, 0: no), 
        # same_route2: whether the second front intruder is on the same route as the ownship (1: yes, 0: no). 
        # If one of the values is None, it means there is no such front intruder.
        # Based on the above information, decide the ownship action: Decelerate / Hold / Accelerate. 
        # Never include reasoning, explanations, or any text other than the single chosen action.
        # """
    
    ownship_spd = ownship_info['speed(m/s)']
    ownship_min_spd = ownship_info['ac_min_spd(m/s)']
    ownship_max_spd = ownship_info['ac_max_spd(m/s)']
    d1 = ownship_info['distance_to_intruder1(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    d2 = ownship_info['distance_to_intruder2(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    o_wpt = ownship_info['ac_dist_to_nxt_wpt(m)']
    dwpt1 = first_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    dwpt2 = second_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_dwpt1 = abs(first_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_dwpt2 = abs(second_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_v1 = first_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_v2 = second_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 2 else None
    same_route1 = ownship_info['intruder1_on_same_route'] if ownship_info['num_intruders_ahead'] >= 1 else None
    same_route2 = ownship_info['intruder2_on_same_route'] if ownship_info['num_intruders_ahead'] >= 2 else None

    def intruder_presence(num_intruders):
        if num_intruders == 0:
            return 'no'
        elif num_intruders == 1:
            return 'one'
        else:
            return 'two'
        
    user_prompt = (
        f"Given the information of the ownship and intruders as follows: \n"
        f"- Ownship: \n"
        f"  - Speed is {speed_indicator(ownship_spd, ownship_max_spd)}. \n"
        f"  - Speed is {desired_speed_indicator(ownship_spd, ownship_max_spd)} than the desired speed. \n"
        f"  - Speed is{' ' if ownship_spd == 0 else ' not '}minimum and is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum.\n"
        # f"- The ownship speed is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum. \n"
        f"  - Distance to the next waypoint is {wypoint_distance_indicator(o_wpt)}.   \n"
        f"  - There are {intruder_presence(ownship_info['num_intruders_ahead'])} intruders ahead. \n"
    )

    if ownship_info['num_intruders_ahead'] >= 1:
        user_prompt += (
            f"- Front Intruder 1: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d1)}. \n"
            f"  - The intruder is{' ' if same_route1 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt1)}. \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt1)} than the ownship. \n"
            # f"  - The intruder speed is {first_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v1, ownship_max_spd)} speed compared to the ownship. \n"
        )    

    if ownship_info['num_intruders_ahead'] >= 2:
        user_prompt += (
            f"- Front Intruder 2: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d2)}. \n"
            f"  - The intruder is{' ' if same_route2 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt2)}. \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt2)} than the ownship. \n"
            # f"  - The intruder speed is {second_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v2, ownship_max_spd)} speed compared to the ownship. \n"
        )

    if ownship_info['num_intruders_ahead'] == 0:
        user_prompt += "Since there are no front intruders detected, make your decision based on the current speed and desired speed.\n"
    
    user_prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"
    user_prompt += f"Your response should start with 'The recommended action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate.\n"

    prompt_response = f"The recommended action is: {ownship_action} \n"

    return system_prompt, user_prompt, prompt_response



In [ ]:
def get_ownship_prompt_rag_integration(ownship_intruder_info):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action'] 

    # system_prompt = """You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, such as merging or intersections points, where other agents (intruders) are approaching as well. 
    #     Based on the information of the ownship and intruders, the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    #     but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    #     suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    #     """
    system_prompt = """You are a human pilot action predictor. You have access to the information available to the pilot at each 
    time step, including the ownship aircraft state and the states of the front intruders. The pilot only changes the ownship 
    speed by taking one of the following actions: 'Accelerate', 'Decelerate', or 'Hold'. 
    The pilot does not unnecessarily decelerate since it might occlude the airspace for other agents behind it. 
    Based on the information, you need to predict the action of the pilot. 
    Your response should start with 'The predicted action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate. 
    Never include reasoning, explanations, or any text other than the sentence including the action. \n"""
        
        # Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
        # v: ownship speed(m/s). The value can be either (low, medium, or high) 
        # is_v_zero: whether the ownship speed is zero (yes, no).
        # v_min: ownship minimum speed(m/s), 
        # v_max: ownship maximum speed(m/s),
        # d1: the absolute distance between the first front intruder (if any) and ownship(m), 
        # d2: the absolute distance between the second front intruder (if any) and ownship(m), 
        # delta_dwpt1: the absolute difference between the distance to the next waypoint between the first front intruder (if any) and ownship(m), 
        # delta_dwpt2: the absolute difference between the distance to the next waypoint between the second front intruder (if any) and ownship(m), 
        # delta_v1: signed speed difference between the first front intruder and ownship(m/s), 
        # delta_v2: signed speed difference between the second front intruder and ownship(m/s), 
        # same_route1: whether the first front intruder is on the same route as the ownship (1: yes, 0: no), 
        # same_route2: whether the second front intruder is on the same route as the ownship (1: yes, 0: no). 
        # If one of the values is None, it means there is no such front intruder.
        # Based on the above information, decide the ownship action: Decelerate / Hold / Accelerate. 
        # Never include reasoning, explanations, or any text other than the single chosen action.
        # """
    
    ownship_spd = ownship_info['speed(m/s)']
    ownship_min_spd = ownship_info['ac_min_spd(m/s)']
    ownship_max_spd = ownship_info['ac_max_spd(m/s)']
    d1 = ownship_info['distance_to_intruder1(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    d2 = ownship_info['distance_to_intruder2(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    o_wpt = ownship_info['ac_dist_to_nxt_wpt(m)']
    dwpt1 = first_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    dwpt2 = second_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_dwpt1 = abs(first_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_dwpt2 = abs(second_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_v1 = first_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_v2 = second_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 2 else None
    same_route1 = ownship_info['intruder1_on_same_route'] if ownship_info['num_intruders_ahead'] >= 1 else None
    same_route2 = ownship_info['intruder2_on_same_route'] if ownship_info['num_intruders_ahead'] >= 2 else None

    def intruder_presence(num_intruders):
        if num_intruders == 0:
            return 'no'
        elif num_intruders == 1:
            return 'one'
        else:
            return 'two'
        
    intruder1_type = "N/A"
    intruder2_type = "N/A"
        
    user_prompt = (
        f"Given the information of the ownship and intruders as follows: \n"
        f"- Ownship: \n"
        f"  - Speed is {speed_indicator(ownship_spd, ownship_max_spd)} ({ownship_spd} m/s), where minimum possible speed is {ownship_min_spd} m/s  and maximum possible speed is {ownship_max_spd} m/s. \n"
        f"  - Speed is {desired_speed_indicator(ownship_spd, ownship_max_spd)} than the desired speed. \n"
        f"  - Speed is{' ' if ownship_spd == 0 else ' not '}minimum and is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum.\n"
        # f"- The ownship speed is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum. \n"
        f"  - Distance to the next waypoint is {wypoint_distance_indicator(o_wpt)} ({o_wpt} m).   \n"
        f"  - There are {intruder_presence(ownship_info['num_intruders_ahead'])} intruders ahead. \n"
    )

    if ownship_info['num_intruders_ahead'] >= 1:
        user_prompt += (
            f"- Front Intruder 1: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d1)} ({d1} m). \n"
            f"  - The intruder is{' ' if same_route1 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt1)} ({dwpt1} m). \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt1)} than the ownship. \n"
            f"  - The intruder speed is {first_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v1, ownship_max_spd)} speed compared to the ownship. \n"
        ) 
        intruder1_type = first_intruder_info['ac_type']   

    if ownship_info['num_intruders_ahead'] >= 2:
        user_prompt += (
            f"- Front Intruder 2: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d2)} ({d2} m). \n"
            f"  - The intruder is{' ' if same_route2 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt2)} ({dwpt2} m). \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt2)} than the ownship. \n"
            f"  - The intruder speed is {second_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v2, ownship_max_spd)} speed compared to the ownship. \n"
        )
        intruder2_type = second_intruder_info['ac_type']

    if ownship_info['num_intruders_ahead'] == 0:
        user_prompt += "Since there are no front intruders detected, make your decision based on the current speed and desired speed.\n"
    
    user_prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"
    user_prompt += f"Your response should start with 'The recommended action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate.\n"

    prompt_response = f"The recommended action is: {ownship_action} \n"

    ownship_type = ownship_info['ac_type']

    intruder1_type_state = 1 if ownship_type == intruder1_type else 0 if ownship_info['num_intruders_ahead'] >=1 else -1

    state_vec = np.array([
        ownship_spd,
        ownship_min_spd,
        ownship_max_spd,
        d1 if d1 is not None else -1,
        d2 if d2 is not None else -1,
        delta_dwpt1 if delta_dwpt1 is not None else -1,
        delta_dwpt2 if delta_dwpt2 is not None else -1,
        delta_v1 if delta_v1 is not None else -1,
        delta_v2 if delta_v2 is not None else -1,
        same_route1 if same_route1 is not None else -1,
        same_route2 if same_route2 is not None else -1,
        1 if ownship_type == "Amazon Prime Air - MK30 Model" else 0,
        1 if intruder1_type == "Amazon Prime Air - MK30 Model" else 0 if intruder1_type == "Google X-Wing" else -1,
        1 if intruder2_type == "Amazon Prime Air - MK30 Model" else 0 if intruder2_type == "Google X-Wing" else -1, 
    ])

    return system_prompt, user_prompt, prompt_response, state_vec

In [ ]:
import numpy as np

def get_ownship_prompt_rag_integration_v2(ownship_intruder_info, info_index, bucket):
    """
    Returns a formatted string prompt for the ownship information.
    """
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action'] 

    # system_prompt = """You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, such as merging or intersections points, where other agents (intruders) are approaching as well. 
    #     Based on the information of the ownship and intruders, the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    #     but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    #     suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    #     """
    system_prompt = """You are a human pilot action predictor. You have access to the information available to the pilot at each 
    time step, including the ownship aircraft state and the states of the front intruders. The pilot only changes the ownship 
    speed by taking one of the following actions: 'Accelerate', 'Decelerate', or 'Hold'. 
    The pilot does not unnecessarily decelerate since it might occlude the airspace for other agents behind it. 
    Based on the information, you need to predict the action of the pilot. 
    Your response should start with 'The predicted action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate. 
    Never include reasoning, explanations, or any text other than the sentence including the action. \n"""
        
        # Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
        # v: ownship speed(m/s). The value can be either (low, medium, or high) 
        # is_v_zero: whether the ownship speed is zero (yes, no).
        # v_min: ownship minimum speed(m/s), 
        # v_max: ownship maximum speed(m/s),
        # d1: the absolute distance between the first front intruder (if any) and ownship(m), 
        # d2: the absolute distance between the second front intruder (if any) and ownship(m), 
        # delta_dwpt1: the absolute difference between the distance to the next waypoint between the first front intruder (if any) and ownship(m), 
        # delta_dwpt2: the absolute difference between the distance to the next waypoint between the second front intruder (if any) and ownship(m), 
        # delta_v1: signed speed difference between the first front intruder and ownship(m/s), 
        # delta_v2: signed speed difference between the second front intruder and ownship(m/s), 
        # same_route1: whether the first front intruder is on the same route as the ownship (1: yes, 0: no), 
        # same_route2: whether the second front intruder is on the same route as the ownship (1: yes, 0: no). 
        # If one of the values is None, it means there is no such front intruder.
        # Based on the above information, decide the ownship action: Decelerate / Hold / Accelerate. 
        # Never include reasoning, explanations, or any text other than the single chosen action.
        # """
    
    ownship_spd = ownship_info['speed(m/s)']
    ownship_min_spd = ownship_info['ac_min_spd(m/s)']
    ownship_max_spd = ownship_info['ac_max_spd(m/s)']
    d1 = ownship_info['distance_to_intruder1(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    d2 = ownship_info['distance_to_intruder2(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    o_wpt = ownship_info['ac_dist_to_nxt_wpt(m)']
    dwpt1 = first_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 1 else None
    dwpt2 = second_intruder_info['ac_dist_to_nxt_wpt(m)'] if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_dwpt1 = abs(first_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_dwpt2 = abs(second_intruder_info['ac_dist_to_nxt_wpt(m)'] - ownship_info['ac_dist_to_nxt_wpt(m)']) if ownship_info['num_intruders_ahead'] >= 2 else None
    delta_v1 = first_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 1 else None
    delta_v2 = second_intruder_info['speed(m/s)'] - ownship_spd if ownship_info['num_intruders_ahead'] >= 2 else None
    same_route1 = ownship_info['intruder1_on_same_route'] if ownship_info['num_intruders_ahead'] >= 1 else None
    same_route2 = ownship_info['intruder2_on_same_route'] if ownship_info['num_intruders_ahead'] >= 2 else None

    def intruder_presence(num_intruders):
        if num_intruders == 0:
            return 'no'
        elif num_intruders == 1:
            return 'one'
        else:
            return 'two'
        
    intruder1_type = "N/A"
    intruder2_type = "N/A"
        
    user_prompt = (
        f"Given the information of the ownship and intruders as follows: \n"
        f"- Ownship: \n"
        f"  - Speed is {speed_indicator(ownship_spd, ownship_max_spd)} ({ownship_spd} m/s), where minimum possible speed is {ownship_min_spd} m/s  and maximum possible speed is {ownship_max_spd} m/s. \n"
        f"  - Speed is {desired_speed_indicator(ownship_spd, ownship_max_spd)} than the desired speed. \n"
        f"  - Speed is{' ' if ownship_spd == 0 else ' not '}minimum and is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum.\n"
        # f"- The ownship speed is{' ' if ownship_spd >= 0.9 * ownship_max_spd else ' not '}maximum. \n"
        f"  - Distance to the next waypoint is {wypoint_distance_indicator(o_wpt)} ({o_wpt} m).   \n"
        f"  - There are {intruder_presence(ownship_info['num_intruders_ahead'])} intruders ahead. \n"
    )

    if ownship_info['num_intruders_ahead'] >= 1:
        user_prompt += (
            f"- Front Intruder 1: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d1)} ({d1} m). \n"
            f"  - The intruder is{' ' if same_route1 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt1)} ({dwpt1} m). \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt1)} than the ownship. \n"
            f"  - The intruder speed is {first_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v1, ownship_max_spd)} speed compared to the ownship. \n"
        ) 
        intruder1_type = first_intruder_info['ac_type']   

    if ownship_info['num_intruders_ahead'] >= 2:
        user_prompt += (
            f"- Front Intruder 2: \n"
            f"  - The euclidean distance to the ownship is {distance_indicator(d2)} ({d2} m). \n"
            f"  - The intruder is{' ' if same_route2 else ' not '}on the same route as the ownship. \n"
            f"  - The intruder distance to the next waypoint is {wypoint_distance_indicator(dwpt2)} ({dwpt2} m). \n"
            f"  - The intruder distance to the next waypoint is {delta_waypoint_distance_indicator(delta_dwpt2)} than the ownship. \n"
            f"  - The intruder speed is {second_intruder_info['speed(m/s)']} m/s. \n"
            f"  - The intruder is moving at a {delta_speed_indicator(delta_v2, ownship_max_spd)} speed compared to the ownship. \n"
        )
        intruder2_type = second_intruder_info['ac_type']

    if ownship_info['num_intruders_ahead'] == 0:
        user_prompt += "Since there are no front intruders detected, make your decision based on the current speed and desired speed.\n"

    similar_action = None
    mask = 1000
    data_length = bucket.shape[0]
    i = info_index
    if i < mask:
        initial_j = 0
        final_j = 2 * mask
    elif mask <= i <= data_length - mask:
        initial_j = i - mask
        final_j = i + mask
    else:
        initial_j =  data_length - 2 * mask
        final_j = data_length
    ownship_bucket = bucket[info_index][initial_j:final_j]
    # remove the (info_index - initial_j)th element from ownship_bucket
    ownship_bucket = np.delete(ownship_bucket, info_index - initial_j)
    # Fine the argmin index
    ownship_argmin_index = np.argmin(ownship_bucket) + initial_j

    similar_action = data[ownship_argmin_index]['ownship_action']

    user_prompt += f"Also, consider that in a similar situation, the pilot took the action: {similar_action}. This action might not be the best choice, but it provides some context.\n"
    
    user_prompt += f"Based on the above information, what actions should the ownship take? (Decelerate/Hold/Accelerate) \n"
    user_prompt += f"Your response should start with 'The recommended action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate.\n"

    prompt_response = f"The predicted action is: {ownship_action} \n"

    ownship_type = ownship_info['ac_type']

    intruder1_type_state = 1 if ownship_type == intruder1_type else 0 if ownship_info['num_intruders_ahead'] >=1 else -1

    state_vec = np.array([
        ownship_spd,
        ownship_min_spd,
        ownship_max_spd,
        d1 if d1 is not None else -1,
        d2 if d2 is not None else -1,
        delta_dwpt1 if delta_dwpt1 is not None else -1,
        delta_dwpt2 if delta_dwpt2 is not None else -1,
        delta_v1 if delta_v1 is not None else -1,
        delta_v2 if delta_v2 is not None else -1,
        same_route1 if same_route1 is not None else -1,
        same_route2 if same_route2 is not None else -1,
        1 if ownship_type == "Amazon Prime Air - MK30 Model" else 0,
        1 if intruder1_type == "Amazon Prime Air - MK30 Model" else 0 if intruder1_type == "Google X-Wing" else -1,
        1 if intruder2_type == "Amazon Prime Air - MK30 Model" else 0 if intruder2_type == "Google X-Wing" else -1, 
    ])

    return system_prompt, user_prompt, prompt_response, state_vec

In [2]:
import json
import pandas as pd

# Load the JSONL file
data = []
dataset_dir = '/data/Iman/datasets/'
with open(dataset_dir + 'raw_ownship_intruders_data_Nov6.jsonl', 'r') as file:
    for line in file:
        data.append(json.loads(line.strip()))

print(f"Loaded {len(data)} rows from the file")

Loaded 39564 rows from the file


In [7]:
def euc_distance(p1, p2):
    p1 = p1 / np.linalg.norm(p1)
    p2 = p2 / np.linalg.norm(p2)
    return np.linalg.norm(p1 - p2)

In [12]:
import numpy as np
bucket = np.zeros((len(data), len(data)))
for i in range(len(data)):
    _, _, _, P1 = get_ownship_prompt_rag_integration(data[i])
    initial_j = 0
    final_j = len(data)
    mask = 1000
    if i < mask:
        initial_j = 0
        final_j = 2 * mask
    elif mask <= i <= len(data) - mask:
        initial_j = i - mask
        final_j = i + mask
    else:
        initial_j = len(data) - 2 * mask
        final_j = len(data)
    for j in range(initial_j, final_j):
        _, _, _, P2 = get_ownship_prompt_rag_integration(data[j])
        dist = euc_distance(P1, P2)
        bucket[i][j] = dist

In [14]:
bucket[0][0:1000]

array([0.        , 0.        , 0.07530587, 0.07530587, 0.14437527,
       0.14437527, 0.17641269, 0.17565731, 0.17641269, 0.17641269,
       0.07096224, 0.17641269, 0.17641269, 0.04758756, 0.17641269,
       0.17641269, 0.1293338 , 0.17641269, 0.17641269, 0.20761436,
       0.17641269, 0.17641269, 0.20761436, 0.17641269, 1.4231508 ,
       0.17641269, 0.20761436, 0.17641269, 1.42337822, 0.17641269,
       0.20761436, 0.17641269, 0.0818837 , 0.17641269, 1.42339838,
       0.20761436, 0.17641269, 0.        , 0.17641269, 1.4232203 ,
       0.20761436, 0.17641269, 0.07530587, 0.17641269, 1.42332767,
       0.20761436, 0.17641269, 0.17641269, 1.4253496 , 0.17641269,
       1.42381104, 0.20761436, 1.43005027, 0.17641269, 1.42488481,
       1.42432664, 1.42428388, 0.20761436, 1.42977289, 0.17641269,
       1.42464734, 1.42386889, 1.42474523, 0.20761436, 1.42939527,
       0.17641269, 1.42439796, 1.42336788, 1.42519445, 0.20761436,
       1.42890814, 0.17641269, 1.42413576, 1.42281716, 1.42563

In [18]:
np.save('bucket.npy', bucket)
print(f"Saved bucket array with shape {bucket.shape} to bucket.npy")

Saved bucket array with shape (39564, 39564) to bucket.npy


In [15]:
_, _, _, state_vec = get_ownship_prompt_rag_integration(data[0])
state_vec

array([15.95,  0.  , 30.87, -1.  , -1.  , -1.  , -1.  , -1.  , -1.  ,
       -1.  , -1.  ,  0.  , -1.  , -1.  ])

In [16]:
_, _, _, state_vec = get_ownship_prompt_rag_integration(data[1])
state_vec

array([15.95,  0.  , 30.87, -1.  , -1.  , -1.  , -1.  , -1.  , -1.  ,
       -1.  , -1.  ,  0.  , -1.  , -1.  ])

In [5]:
for row in data[1000:1001]:
    print(row)

{'ownship_info': {'agent_id': 'AB0003', 'ac_type': 'Google X-Wing', 'ac_lat': 33.14941, 'ac_lon': -96.850262, 'next_wpt_id': 'WP9', 'next_wpt_type': 'Regular', 'ac_dist_to_nxt_wpt(m)': 3806.98, 'speed(m/s)': 25.72, 'ac_min_spd(m/s)': 0.0, 'ac_max_spd(m/s)': 30.87, 'speed_change_per_second(m/s2)': 1.03, 'heading(deg)': 272.66, 'heading_scale': '{North=0, East=90, South=180, West=270}', 'altitude(m)': 1372.64, 'ac_route_id': 'R_2', 'ac_last_action': 'accelerate', 'num_intruders_ahead': 2, 'ac_desired_spd(m/s)': 23.15, 'time_to_collision_with_inruder1(s)': inf, 'intruder1_on_same_route': True, 'did_ownship_have_NMAC': False, 'time_to_collision_with_inruder2(s)': 1356.5167547702029, 'intruder2_on_same_route': True, 'distance_to_intruder1(m)': 1440.0, 'distance_to_intruder2(m)': 698.2}, 'first_closest_front_intruder_info': {'agent_id': 'AB0002', 'ac_type': 'Google X-Wing', 'ac_lat': 33.15001, 'ac_lon': -96.865713, 'next_wpt_id': 'WP9', 'next_wpt_type': 'Regular', 'ac_dist_to_nxt_wpt(m)': 23

In [17]:
sys_prompt, user_prompt, prompt_response, state_vec = get_ownship_prompt_rag_integration(data[1000])
print(f"### System:\n{sys_prompt.strip()}")
print(f"### User:\n{user_prompt.strip()}")
print(f"### Response:\n{prompt_response.strip()}")

### System:
You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, such as merging or intersections points, where other agents (intruders) are approaching as well.
Based on the information of the ownship and intruders, the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, but there might be other intruders behind the ownship. 
Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by suggesting appropriate speed adjustments. 
The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it. 
Your response should start with 'The recommended action is: ' followed by one of the actions: Decelerate, Hold, or Accelerate.
### User:
Given the information of the ownship and intruders as follows: 
- Ownship: 
  - Speed is very high (23.66 m/s), where minimum possible speed

In [73]:
with open('train_data_new_final_prompt_no_number.jsonl', 'w') as outfile:
    for row in data:  
        prompt = {}
        system_prompt, user_prompt, prompt_response = get_ownship_prompt_no_number(row)
        prompt['system_prompt'] = system_prompt
        prompt['user_prompt'] = user_prompt
        prompt['prompt_response'] = prompt_response
        outfile.write(json.dumps(prompt) + '\n')

print("Finished writing to train_data_new_final_prompt_no_number.jsonl")



Finished writing to train_data_new_final_prompt_no_number.jsonl


In [71]:
with open('train_data_new_final_prompt_with_number.jsonl', 'w') as outfile:
    for row in data:  
        prompt = {}
        system_prompt, user_prompt, prompt_response = get_ownship_prompt(row)
        prompt['system_prompt'] = system_prompt
        prompt['user_prompt'] = user_prompt
        prompt['prompt_response'] = prompt_response
        outfile.write(json.dumps(prompt) + '\n')

print("Finished writing to train_data_new_final_prompt_with_number.jsonl")



Finished writing to train_data_new_final_prompt_with_number.jsonl


In [ ]:
def new_prompt(ownship_intruder_info):
    ownship_info = ownship_intruder_info['ownship_info']
    first_intruder_info = ownship_intruder_info['first_closest_front_intruder_info']
    second_intruder_info = ownship_intruder_info['second_closest_front_intruder_info']
    ownship_action = ownship_intruder_info['ownship_action']


    


In [7]:
for row in data[:1]:  # Display first 5 rows
    system_prompt, user_prompt, prompt_response = get_ownship_prompt(env, row)
    print(system_prompt)
    print(user_prompt)
    print(prompt_response)

You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, 
    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, 
    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
    v: ownship speed(m/s), 
    v_min: ownship minimum speed(m/s), 
    v_max: ownship maximum speed(m/s),
    d1: the absolute distance bet

In [5]:
with open('ownship_prompts_data_Nov10.jsonl', 'w') as outfile:
    for row in data:  
        prompt = {}
        system_prompt, user_prompt, prompt_response = get_ownship_prompt(env, row)
        prompt['system_prompt'] = system_prompt
        prompt['user_prompt'] = user_prompt
        prompt['prompt_response'] = prompt_response
        outfile.write(json.dumps(prompt) + '\n')